# AccentShift â€” Seed-VC Fine-Tuning on Kaggle

**Before running:**
1. Settings (right panel) â†’ **Accelerator: GPU T4 x2** or P100
2. Settings â†’ **Internet: On**
3. Add-ons â†’ **Secrets** â†’ add `TELEGRAM_TOKEN` and `TELEGRAM_CHAT_ID` (optional but recommended)
4. Run All

Expected total time: ~2â€“3 hours on T4, ~3â€“5 hours on P100.

In [ ]:
# Cell 1: Verify GPU
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
# Cell 2: Telegram setup (reads from Kaggle Secrets)
# Add-ons â†’ Secrets â†’ add TELEGRAM_TOKEN and TELEGRAM_CHAT_ID before running
import os
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['TELEGRAM_TOKEN'] = secrets.get_secret('TELEGRAM_TOKEN')
    os.environ['TELEGRAM_CHAT_ID'] = secrets.get_secret('TELEGRAM_CHAT_ID')
    print('Telegram secrets loaded OK')
except Exception as e:
    print(f'Telegram secrets not found ({e}) â€” training will run without notifications')

In [ ]:
# Cell 2b: Test Telegram BEFORE training — verify notifications work
import os, urllib.request, urllib.parse

token = os.environ.get('TELEGRAM_TOKEN', '').strip()
chat_id = os.environ.get('TELEGRAM_CHAT_ID', '').strip()

if not token or not chat_id:
    print('No Telegram secrets found. Run Cell 2 first, or skip.')
else:
    url = f'https://api.telegram.org/bot{token}/sendMessage'
    msg = 'AccentShift Kaggle — Telegram test OK! Training will notify here.'
    data = urllib.parse.urlencode({'chat_id': chat_id, 'text': msg}).encode()
    try:
        urllib.request.urlopen(url, data=data, timeout=10)
        print('Telegram test PASSED — check your phone!')
    except Exception as e:
        print(f'Telegram FAILED: {e}')
        print('Check: token has no spaces, chat_id is correct integer')


In [ ]:
%%bash
# Cell 3: Clone repo
echo '=== Cloning AccentShift repo ==='
git clone --depth 1 https://github.com/nischal2805/AccentShift.git /kaggle/working/AccentShift
cd /kaggle/working/AccentShift
git checkout div
echo "Branch: $(git branch --show-current)"
echo "Latest commit: $(git log --oneline -1)"
echo '=== Repo ready ==='

In [ ]:
%%bash
# Cell 4: Install pipeline deps (torch already on Kaggle)
echo '=== Installing deps ==='
pip install -q \
    "librosa>=0.10.2" soundfile pyloudnorm pyworld silero-vad \
    transformers jiwer speechbrain scikit-learn joblib pyyaml click \
    huggingface-hub tqdm hydra-core omegaconf einops munch \
    accelerate pydub tensorboard gdown 2>&1 | tail -5
echo '=== Deps installed ==='

In [ ]:
%%bash
# Cell 5: Clone Seed-VC + Amphion + install seed-vc requirements
cd /kaggle/working/AccentShift/backend
mkdir -p third_party

echo '=== Cloning Seed-VC ==='
if [ ! -d third_party/seed-vc ]; then
    git clone --depth 1 --filter=blob:none --single-branch \
        https://github.com/Plachtaa/seed-vc.git third_party/seed-vc
    echo 'Seed-VC cloned'
else
    echo 'Seed-VC already present'
fi

echo '=== Installing Seed-VC deps ==='
pip install -q -r third_party/seed-vc/requirements.txt --no-deps 2>/dev/null || true
echo 'Seed-VC deps done'

echo '=== Cloning Amphion ==='
if [ ! -d third_party/Amphion ]; then
    git clone --depth 1 --filter=blob:none --single-branch \
        https://github.com/open-mmlab/Amphion.git third_party/Amphion
    echo 'Amphion cloned'
else
    echo 'Amphion already present'
fi

echo '=== third_party ready ==='

In [ ]:
%%bash
# Cell 6: Download L2-Arctic v5.0 from Google Drive (~7GB)
echo '=== Downloading L2-Arctic dataset ==='
mkdir -p /kaggle/working/AccentShift/backend/data/l2arctic_raw
ZIP=/kaggle/working/AccentShift/backend/data/l2arctic_raw/l2arctic_v5.zip

if [ ! -f "$ZIP" ]; then
    gdown 'https://drive.google.com/uc?id=1ciCw_ttbw7a9r7d5DZzTJwoZq5rQB3TA' -O "$ZIP"
else
    echo 'ZIP already downloaded, skipping'
fi
echo "ZIP size: $(du -sh $ZIP)"

In [ ]:
%%bash
# Cell 7: Unzip + organize WAVs by accent
# Deletes zips after extract to stay within 20GB Kaggle disk limit
cd /kaggle/working/AccentShift/backend
RAW=data/l2arctic_raw

echo '=== Unzipping main archive ==='
unzip -o -q $RAW/l2arctic_v5.zip -d $RAW
echo 'Main zip extracted'

echo '=== Freeing space: deleting main zip ==='
rm -f $RAW/l2arctic_v5.zip
df -h /kaggle/working

echo '=== Unzipping per-speaker archives (deleting each after extract) ==='
for spk_zip in $RAW/*.zip; do
    [ -f "$spk_zip" ] || continue
    spk=$(basename "$spk_zip" .zip)
    echo "  $spk..."
    unzip -o -q "$spk_zip" -d $RAW
    rm -f "$spk_zip"
done
echo 'All speakers extracted'
df -h /kaggle/working

echo '=== Organizing WAVs by accent ==='
python3 - << 'EOF'
from pathlib import Path
import shutil

SPEAKER_ACCENT = {
    'ASI':'indian_english','RRBI':'indian_english','SVBI':'indian_english','TNI':'indian_english',
    'BWC':'chinese_english','LXC':'chinese_english','NCC':'chinese_english','TXHC':'chinese_english',
    'HJK':'korean_english','HKK':'korean_english','YDCK':'korean_english','YKWK':'korean_english',
    'HQTV':'vietnamese_english','PNV':'vietnamese_english','THV':'vietnamese_english','TLV':'vietnamese_english',
    'EBVS':'spanish_english','ERMS':'spanish_english','MBMPS':'spanish_english','NJS':'spanish_english',
    'ABA':'arabic_english','SKA':'arabic_english','YBAA':'arabic_english','ZHAA':'arabic_english',
}

raw = Path('/kaggle/working/AccentShift/backend/data/l2arctic_raw')
ft = Path('/kaggle/working/AccentShift/backend/data/finetune')

for spk, accent in SPEAKER_ACCENT.items():
    out = ft / accent
    out.mkdir(parents=True, exist_ok=True)
    # Try multiple layout candidates
    found = 0
    for candidate in [raw/spk/'wav', raw/spk, raw/'l2arctic_v5'/spk/'wav', raw/'l2arctic_v5'/spk]:
        if candidate.is_dir():
            for wav in candidate.rglob('*.wav'):
                dst = out / wav.name
                if not dst.exists():
                    shutil.copy2(wav, dst)
                found += 1
            break
    print(f'  {spk} -> {accent} ({found} WAVs)')

print('')
for d in sorted(ft.iterdir()):
    if d.is_dir():
        count = len(list(d.glob('*.wav')))
        print(f'  {d.name}: {count} WAVs')
EOF


In [ ]:
# Cell 8: TRAIN — ~2-3 hours on T4
# fp16 = T4/P100 compatible. Uses Python subprocess for live log streaming.
# First step downloads HuBERT + CAMPPlus (~2GB) — normal, not a hang.
import subprocess, sys, os

os.chdir('/kaggle/working/AccentShift/backend')
os.environ['HF_HOME'] = '/kaggle/working/AccentShift/backend/.hf_cache'
# TELEGRAM_TOKEN and TELEGRAM_CHAT_ID already in os.environ from Cell 2

print(f'=== Starting training at {__import__("datetime").datetime.now().strftime("%H:%M:%S")} ===')
print('TELEGRAM_TOKEN set:', bool(os.environ.get('TELEGRAM_TOKEN','').strip()))
print('TELEGRAM_CHAT_ID set:', bool(os.environ.get('TELEGRAM_CHAT_ID','').strip()))
print('')

cmd = [
    sys.executable, 'scripts/finetune_style.py',
    '--accent', 'all',
    '--steps', '15000',
    '--batch-size', '8',
    '--save-every', '1000',
    '--num-workers', '2',
    '--mixed-precision', 'fp16',
]

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd='/kaggle/working/AccentShift/backend',
    env=os.environ.copy(),
)

for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
print(f'
=== Training exit code: {proc.returncode} ===')
print(f'=== Finished at {__import__("datetime").datetime.now().strftime("%H:%M:%S")} ===')


In [ ]:
%%bash
# Cell 9: Verify checkpoints
echo '=== Checkpoints ==='
find /kaggle/working/AccentShift/backend/runs/ -name '*.pth' -exec du -sh {} \;
echo ''
echo '=== Run directory listing ==='
ls -lh /kaggle/working/AccentShift/backend/runs/*/ 2>/dev/null || echo 'No runs/ directory found'

In [ ]:
import shutil, os
# Cell 10: Zip checkpoint for download
runs_dir = '/kaggle/working/AccentShift/backend/runs/'
out_zip = '/kaggle/working/seedvc_finetuned'
shutil.make_archive(out_zip, 'zip', runs_dir)
size = os.path.getsize(out_zip + '.zip') / (1024**2)
print(f'Checkpoint zipped: {out_zip}.zip ({size:.0f} MB)')
print('Download via Files panel on the right ->')